# 01 — Data Exploration

This notebook performs the initial exploration of the customer feedback dataset.

The goal of this stage is to understand:

- Dataset size and structure
- Available variables
- Distribution of treatment vs control groups
- Presence of VOLT vs non-VOLT segments
- Structure of open-text customer comments

This step helps guide the subsequent NLP and modeling pipeline.

## Import Required Libraries

We begin by importing the libraries required for data loading and exploration.

In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

## Load and Combine Experiment Data

The Excel dataset contains two sheets:

- **C Control** → customers exposed to the original script
- **C Pilot** → customers exposed to the new decision-support script

We load both sheets and label them before combining the datasets.

In [26]:
control = pd.read_excel(
    "../data/raw/customer_feedback_script.xlsx",
    sheet_name="C Control"
)

pilot = pd.read_excel(
    "../data/raw/customer_feedback_script.xlsx",
    sheet_name="C Pilot"
)

control["experiment_group"] = "control"
pilot["experiment_group"] = "treatment"

df = pd.concat([control, pilot], ignore_index=True)

df.shape

(647, 14)

## Dataset Overview

In this section we examine the basic structure of the dataset, including:

- number of observations
- number of variables
- column names
- missing values
- data types

In [27]:
df["experiment_group"].value_counts()

experiment_group
control      380
treatment    267
Name: count, dtype: int64

In [28]:
# checking volt customers
df["VOLT_FLAG"].value_counts(dropna=False)

VOLT_FLAG
NaN    368
yes    279
Name: count, dtype: int64

In [29]:
#checking usable comments
df["LTR_COMMENT"].notna().sum()

np.int64(482)

In [30]:
df = df.drop(columns=["COLUMN_4"])
df.columns

Index(['VOLT_FLAG', 'SURVEY_ID', 'SCORE', 'LTR_COMMENT', 'PRIMARY_REASON',
       'TO_CHAR', 'CONNECTION_TIME', 'SALES_PERSON_SAT', 'SALES_FRIENDLY_SAT',
       'COMMINICATION_SAT', 'FIRST_BILL_SAT', 'AGENT_KNOWLEDGE',
       'experiment_group'],
      dtype='str')

In [31]:
df.shape

(647, 13)

## Rename Date Column

The column `TO_CHAR` represents the date of the survey response.
For readability, we rename it to `date`.

In [32]:
df = df.rename(columns={"TO_CHAR": "date"})
df.columns

Index(['VOLT_FLAG', 'SURVEY_ID', 'SCORE', 'LTR_COMMENT', 'PRIMARY_REASON',
       'date', 'CONNECTION_TIME', 'SALES_PERSON_SAT', 'SALES_FRIENDLY_SAT',
       'COMMINICATION_SAT', 'FIRST_BILL_SAT', 'AGENT_KNOWLEDGE',
       'experiment_group'],
      dtype='str')

In [33]:
df["date"].dtype

dtype('O')

In [34]:
## Inspect Invalid Date Values
df["date"].unique()[:20]

array([Timestamp('2023-02-01 00:00:00'), Timestamp('2023-03-01 00:00:00'),
       Timestamp('2023-04-01 00:00:00'), Timestamp('2023-05-01 00:00:00'),
       Timestamp('2023-06-01 00:00:00'), '01/03/2company3',
       '01/04/2company3', '01/05/2company3', '01/06/2company3'],
      dtype=object)

## Clean Malformed Date Strings

Some date values contain a text error (`2company3` instead of `2023`).
We correct this issue before converting the column fully to datetime.

In [35]:
df["date"] = df["date"].astype(str).str.replace("2company3", "2023", regex=False)
df["date"].unique()[:10]

<StringArray>
['2023-02-01 00:00:00', '2023-03-01 00:00:00', '2023-04-01 00:00:00',
 '2023-05-01 00:00:00', '2023-06-01 00:00:00',          '01/03/2023',
          '01/04/2023',          '01/05/2023',          '01/06/2023']
Length: 9, dtype: str

In [36]:
df["date"] = pd.to_datetime(df["date"], format="mixed")
df["month"] = df["date"].dt.month
df["month"].value_counts().sort_index()

month
1    247
2     65
3    134
4     90
5     93
6     18
Name: count, dtype: int64

## Month Distribution

We examine the distribution of survey responses across months
to understand the time coverage of the experiment.

In [37]:
df["month"].value_counts().sort_index()

month
1    247
2     65
3    134
4     90
5     93
6     18
Name: count, dtype: int64

In [38]:
df["month_name"] = df["date"].dt.month_name()
df["month_name"].value_counts()

month_name
January     247
March       134
May          93
April        90
February     65
June         18
Name: count, dtype: int64

In [39]:
## Sort Month Names Chronologically
month_order = ["January", "February", "March", "April", "May", "June"]

df["month_name"] = pd.Categorical(
    df["month_name"],
    categories=month_order,
    ordered=True
)

df["month_name"].value_counts().sort_index()

month_name
January     247
February     65
March       134
April        90
May          93
June         18
Name: count, dtype: int64

## Experiment Group Distribution

We examine the number of observations belonging to the control and treatment groups.
This confirms whether the experiment groups are balanced.

In [40]:
df["experiment_group"].value_counts()

experiment_group
control      380
treatment    267
Name: count, dtype: int64

## Experiment Groups Across Months

To understand the experiment rollout, we examine how control and treatment
observations are distributed across months.

In [41]:
pd.crosstab(df["month_name"], df["experiment_group"])

experiment_group,control,treatment
month_name,,
January,0,247
February,45,20
March,134,0
April,90,0
May,93,0
June,18,0


This means:
- Pilot phase      → January
- Experiment phase → February
- Post experiment  → March–June

In [42]:
df["experiment_group"].value_counts()

experiment_group
control      380
treatment    267
Name: count, dtype: int64

In [43]:
df[df["experiment_group"] == "treatment"]["month_name"].value_counts()

month_name
January     247
February     20
March         0
April         0
May           0
June          0
Name: count, dtype: int64

## Extract February Experiment Window

February is the only month where both control and treatment
groups appear simultaneously. We isolate this subset to perform
a fair comparison between the two scripts.

In [44]:
feb_df = df[df["month_name"] == "February"]

feb_df["experiment_group"].value_counts()

experiment_group
control      45
treatment    20
Name: count, dtype: int64

## Score Distribution During Experiment Window (February)

We compare customer scores between the control and treatment groups
during the February experiment period to see whether the new script
may have improved customer outcomes.

In [45]:
feb_df.groupby("experiment_group")["SCORE"].describe()

,count,mean,std,min,25%,50%,75%,max
experiment_group,,,,,,,,
control,45.0,8.066667,3.193744,0.0,8.00,10.0,10.0,10.0
treatment,20.0,7.200000,3.707744,0.0,4.75,9.0,10.0,10.0


## Save Cleaned Dataset

After completing preprocessing and data validation,
we save the cleaned dataset to the processed data folder
so that downstream analysis notebooks can load it directly.

In [47]:
df.to_csv("../data/processed/experiment_data_cleaned.csv", index=False)